# Crop Recommendation System — Full Pipeline
## Mirrors: `build_model.py`, `app.py`, and the Flask web app

This notebook reproduces **exactly** what happens in the project:
1. Load `Crop_recommendation.csv` (7 soil/climate features → 22 crop labels)
2. EDA: distributions, correlation heatmap, crop balance
3. Train 4 models: Decision Tree, Random Forest, KNN, Logistic Regression
4. Compare accuracies, deep-dive the best model
5. Charts: confusion matrix, feature importance, per-crop accuracy
6. Save `model.pkl` — same format loaded by `app.py`
7. Demo prediction with sample inputs

---
## 1. Setup & Load Data

**Dataset**: `Crop_recommendation.csv`
- **Features**: N, P, K, temperature, humidity, ph, rainfall
- **Target**: label (crop name — 22 classes)
- **Source**: The same CSV used by `build_model.py` in the project

Upload the CSV to Colab or mount your Google Drive.

In [ ]:
# Install (only if needed — Colab includes these by default)
# !pip install pandas numpy scikit-learn matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_recall_fscore_support
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
print('Dependencies loaded!')

In [ ]:
# Upload the dataset:
# Option A: Upload via Colab UI, then update path below
# Option B: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# CSV_PATH = '/content/drive/MyDrive/Crop_recommendation.csv'

# Option C: Download from a direct URL (replace with your link)
# !wget -q -O Crop_recommendation.csv "<YOUR_GITHUB_RAW_URL>"
# CSV_PATH = 'Crop_recommendation.csv'

CSV_PATH = 'Crop_recommendation.csv'  # Update this to your actual path

df = pd.read_csv(CSV_PATH)
print(f'Dataset shape: {df.shape}')
print(f'{len(df)} rows, {df.shape[1]} columns')
print(f'Features: {[c for c in df.columns if c != "label"]}')
print(f'Target: label ({df["label"].nunique()} unique crops)')
print(f'\nCrops: {sorted(df["label"].unique())}')
print(f'\nCrop distribution:\n{df["label"].value_counts()}')

In [ ]:
# Data overview
print(df.info())
print('\n--- Statistics ---')
print(df.describe())
print('\n--- Missing Values ---')
print(df.isnull().sum())

---
## 2. Exploratory Data Analysis

Visualize the data the same way the project does — plus extra charts for deeper insight.

In [ ]:
# 2a. Crop Class Balance
plt.figure(figsize=(14, 5))
crop_counts = df['label'].value_counts()
sns.barplot(x=crop_counts.index, y=crop_counts.values, palette='viridis')
plt.title('Crop Class Distribution', fontsize=14)
plt.xlabel('Crop')
plt.ylabel('Count')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.savefig('crop_distribution.png', dpi=150)
plt.show()

In [ ]:
# 2b. Correlation Heatmap (same as build_model.py)
plt.figure(figsize=(10, 8))
numeric_cols = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_cols.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
# 2c. Feature Distributions (violin plots split by top 5 crops)
features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
top_crops = df['label'].value_counts().head(5).index.tolist()
top_df = df[df['label'].isin(top_crops)]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for i, feat in enumerate(features):
    sns.violinplot(data=top_df, x='label', y=feat, ax=axes[i], palette='Set2')
    axes[i].set_title(f'{feat} by Crop', fontsize=11)
    axes[i].tick_params(axis='x', rotation=45)
axes[-1].axis('off')
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150)
plt.show()

In [ ]:
# 2d. Pair-wise scatter (N vs K, temperature vs rainfall) colored by crop
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for crop in sorted(df['label'].unique()):
    sub = df[df['label'] == crop]
    axes[0].scatter(sub['N'], sub['K'], label=crop, alpha=0.4, s=10)
    axes[1].scatter(sub['temperature'], sub['rainfall'], label=crop, alpha=0.4, s=10)
axes[0].set_xlabel('Nitrogen (N)')
axes[0].set_ylabel('Potassium (K)')
axes[0].set_title('N vs K by Crop')
axes[1].set_xlabel('Temperature (°C)')
axes[1].set_ylabel('Rainfall (mm)')
axes[1].set_title('Temperature vs Rainfall by Crop')
axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)
plt.tight_layout()
plt.savefig('scatter_pairs.png', dpi=150)
plt.show()

---
## 3. Model Training

Exact same pipeline as `build_model.py`:
1. Extract 7 features: N, P, K, temperature, humidity, ph, rainfall
2. 80/20 train-test split with `random_state=42`
3. Train 4 models: Decision Tree, Random Forest, KNN, Logistic Regression
4. Pick the best by accuracy

In [ ]:
# Extract features and target
X = df[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']]
y = df['label']

# Train-test split (identical to build_model.py)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Training: {X_train.shape[0]} samples')
print(f'Testing:  {X_test.shape[0]} samples')
print(f'Features: {list(X.columns)}')

In [ ]:
# Train 4 models (same as build_model.py)
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42)
}

accuracies = {}
trained_models = {}
best_model = None
best_acc = 0

print('Training models...\n')
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    accuracies[name] = acc
    trained_models[name] = (model, y_pred)
    print(f'{name:25s} Accuracy: {acc:.4f} ({acc*100:.2f}%)')
    
    if acc > best_acc:
        best_acc = acc
        best_model = model

print(f'\nBest Model: {[k for k, v in accuracies.items() if v == best_acc][0]} — {best_acc:.4f}')

---
## 4. Results & Visualizations

#### 4a. Model Accuracy Comparison (same chart the project saves)

In [ ]:
# Accuracy comparison bar chart
plt.figure(figsize=(10, 6))
bars = sns.barplot(x=list(accuracies.keys()), y=list(accuracies.values()), 
                   hue=list(accuracies.keys()), palette='viridis', legend=False)
plt.title('Model Accuracy Comparison', fontsize=14)
plt.ylabel('Accuracy')
plt.ylim(0, 1.1)
for i, (name, acc) in enumerate(accuracies.items()):
    plt.text(i, acc + 0.015, f'{acc:.4f}', ha='center', fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig('accuracy_comparison.png', dpi=150)
plt.show()

#### 4b. Best Model — Confusion Matrix & Classification Report
This is the exact confusion matrix the project saves to `static/confusion_matrix.png`

In [ ]:
best_name = [k for k, v in accuracies.items() if v == best_acc][0]
_, y_pred_best = trained_models[best_name]

print(f'--- {best_name} Classification Report ---')
print(classification_report(y_test, y_pred_best, zero_division=0))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best, labels=best_model.classes_)
plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', linewidths=0.5,
            xticklabels=best_model.classes_, yticklabels=best_model.classes_)
plt.title(f'Confusion Matrix — {best_name}', fontsize=14)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

#### 4c. Per-Crop Accuracy Breakdown

In [ ]:
# Per-crop precision (accuracy proxy)
prec, rec, f1, sup = precision_recall_fscore_support(
    y_test, y_pred_best, labels=sorted(best_model.classes_), zero_division=0
)

crop_acc = pd.DataFrame({
    'Crop': sorted(best_model.classes_),
    'Precision': np.round(prec, 4),
    'Recall': np.round(rec, 4),
    'F1-Score': np.round(f1, 4),
    'Support': sup
}).sort_values('F1-Score', ascending=True)

plt.figure(figsize=(10, 8))
sns.barplot(data=crop_acc, x='F1-Score', y='Crop', palette='RdYlGn')
plt.title(f'Per-Crop F1-Score — {best_name}', fontsize=14)
plt.xlabel('F1-Score')
plt.tight_layout()
plt.savefig('per_crop_accuracy.png', dpi=150)
plt.show()

print(crop_acc.to_string(index=False))

#### 4d. Feature Importance (same as `static/feature_importance.png`)

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    plt.figure(figsize=(9, 6))
    sorted_idx = np.argsort(best_model.feature_importances_)
    sns.barplot(x=best_model.feature_importances_[sorted_idx], 
                y=X.columns[sorted_idx], palette='magma')
    plt.title(f'Feature Importance — {best_name}', fontsize=14)
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=150)
    plt.show()

    for feat, imp in zip(X.columns, best_model.feature_importances_):
        print(f'{feat:15s}: {imp:.4f}')
else:
    print(f'{best_name} does not have feature_importances_ attribute.')

---
## 5. Save Model

Saves `model.pkl` in the **exact same format** that `app.py` loads:
- The model object supports `.predict_proba()` and `.classes_`
- `app.py` reads it with `pickle.load()` and calls `model.predict_proba(features)`

In [ ]:
with open('model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

import os
size_kb = os.path.getsize('model.pkl') / 1024
print(f'Model saved: model.pkl ({best_name}, {best_acc:.4f}, {size_kb:.1f} KB)')

---
## 6. Demo Predictions

Test with the same input ranges the web app accepts (`app.py` lines 24-31):
- N: 0–150, P: 0–150, K: 0–210
- temperature: 5–50°C, humidity: 10–100%
- ph: 0–14, rainfall: 10–350mm

In [ ]:
def predict_crop(N, P, K, temperature, humidity, ph, rainfall, top_k=3):
    features = np.array([[N, P, K, temperature, humidity, ph, rainfall]])
    probabilities = best_model.predict_proba(features)[0]
    crop_probs = sorted(zip(best_model.classes_, probabilities), 
                         key=lambda x: x[1], reverse=True)
    
    print(f'Input: N={N}, P={P}, K={K}, Temp={temperature}°C, '
          f'Humidity={humidity}%, pH={ph}, Rainfall={rainfall}mm\n')
    print(f'Top {top_k} Predictions:')
    for i, (crop, prob) in enumerate(crop_probs[:top_k]):
        conf = prob * 100
        bar = '█' * int(conf / 2) + '░' * (50 - int(conf / 2))
        print(f'  #{i+1} {crop.capitalize():20s}  {conf:6.2f}%  |{bar}|')
    print()

# Test cases from the project's "Test Cases for Model" PDF context + typical values
print('=== Sample Predictions ===\n')
predict_crop(N=90, P=42, K=43, temperature=20.9, humidity=82.0, ph=6.5, rainfall=202.9)    # Sample rice-like
predict_crop(N=100, P=30, K=40, temperature=25.0, humidity=70.0, ph=6.0, rainfall=150.0)  # Moderate
predict_crop(N=50, P=50, K=30, temperature=22.0, humidity=60.0, ph=7.0, rainfall=100.0)   # Low rainfall
predict_crop(N=120, P=60, K=50, temperature=30.0, humidity=85.0, ph=5.5, rainfall=300.0)  # High rainfall

---
## 7. Cross-Validation

5-fold stratified cross-validation for a more reliable accuracy estimate.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(best_model, X, y, cv=cv, scoring='accuracy')

print(f'5-Fold Cross-Validation Scores: {np.round(cv_scores, 4)}')
print(f'Mean: {cv_scores.mean():.4f}  Std: {cv_scores.std():.4f}')

plt.figure(figsize=(8, 4))
sns.barplot(x=[f'Fold {i+1}' for i in range(5)], y=np.round(cv_scores, 4), 
            palette='Blues_d')
plt.axhline(y=cv_scores.mean(), color='red', linestyle='--', 
            label=f'Mean: {cv_scores.mean():.4f}')
plt.title(f'5-Fold CV Accuracy — Best Model', fontsize=14)
plt.ylabel('Accuracy')
plt.ylim(0.85, 1.05)
plt.legend()
plt.tight_layout()
plt.savefig('cross_validation.png', dpi=150)
plt.show()

---
## 8. Dataset Summary Table

Quick reference of the features used by the model.

In [ ]:
summary = pd.DataFrame({
    'Feature': X.columns,
    'Min': X.min(),
    'Max': X.max(),
    'Mean': np.round(X.mean(), 2),
    'Std': np.round(X.std(), 2)
})
print(summary.to_string(index=False))

print(f'\nTotal crops: {y.nunique()}')
print(f'Total samples: {len(df)}')
print(f'Best model: {best_name} ({best_acc:.4f})')